In [3]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import os
from tqdm import tqdm
import torchvision.transforms.v2 as T

# Import utilities from the other files
from data_utils_mask import InstanceSegmentationDataset, collate_fn
from model_utils_mask import get_model

# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4 # Swin Transformers can also be memory intensive
NUM_EPOCHS = 50
LR = 1e-4

# IMPORTANT: Define the number of classes for the model.
# This MUST include the background class.
# We chose 3 classes in data_utils: "Built-Up", "Informal Settlements", "Water"
# So, 3 classes + 1 background = 4
NUM_CLASSES = 7 

# --- Data Path Placeholders ---
BASE_PATH = "../Dataset/Prepared_Dataset"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_PATH, "train/masks")
VAL_IMG_DIR = os.path.join(BASE_PATH, "val/images")
VAL_MASK_DIR = os.path.join(BASE_PATH, "val/masks")

# --- Augmentations ---
transform = T.Compose([
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- Training and Validation Functions ---
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    total_loss = 0
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} (Train)")
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        progress_bar.set_postfix(loss=losses.item())
        
    return total_loss / len(data_loader)

@torch.no_grad()
def validate_epoch(model, data_loader, device):
    model.train() # Validation loss calculation requires train mode for Mask R-CNN
    total_loss = 0
    
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
        
    return total_loss / len(data_loader)

# --- Training Session Function ---
def run_training_session(enhancement_mode, model_save_path):
    print("\n" + "="*60 + f"\nSTARTING TRAINING: Mask R-CNN, Enhancement='{enhancement_mode}'\n" + "="*60 + "\n")

    train_dataset = InstanceSegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=transform, use_enhancement=enhancement_mode)
    val_dataset = InstanceSegmentationDataset(VAL_IMG_DIR, VAL_MASK_DIR, transform=transform, use_enhancement='none')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn)
    
    model = get_model(num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, mode='min')

    best_val_loss = float('inf')
    
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, epoch)
        val_loss = validate_epoch(model, val_loader, DEVICE)
        scheduler.step(val_loss)

        print(f"Epoch {epoch} finished. Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_save_path)
            print(f"Model saved to {model_save_path}! (Best Val Loss: {best_val_loss:.4f})")

# --- Main Execution ---
def main():
    if not os.path.exists(TRAIN_IMG_DIR) or not os.path.exists(VAL_IMG_DIR):
        print(f"!! ERROR: Data paths incorrect. Expected Prepared_Dataset directory at: {BASE_PATH}")
        return

    # --- Mask R-CNN with Swin Transformer Backbone Training Sessions ---
    print("--- Starting Training for Mask R-CNN with Swin-S Backbone ---")
    
    run_training_session('none', "maskrcnn_swin_s_none_50.pth")
    run_training_session('all', "maskrcnn_swin_s_preprocess_50.pth")
    run_training_session('hybrid', "maskrcnn_swin_s_hybrid_50.pth")
    
    print("\nAll requested training sessions completed.")

if __name__ == '__main__':
    try:
        main()
    except Exception as e:
        print(f"\nAn unexpected and fatal error occurred: {e}")

--- Starting Training for Mask R-CNN with Swin-S Backbone ---

STARTING TRAINING: Mask R-CNN, Enhancement='none'

Mask R-CNN model created with Swin-S FPN backbone for 7 classes.


Epoch 1/50 (Train):   0%|          | 0/1782 [00:21<?, ?it/s]


An unexpected and fatal error occurred: Given groups=1, weight of size [256, 768, 1, 1], expected input[4, 25, 25, 768] to have 768 channels, but got 25 channels instead
